# Logging in Python — Your Program's Diary

> **Goal:** replace mystery and random `print()` calls with useful, searchable records of what a program did.

## Explain it like I am 5

Logging is a diary for your program. The program writes, “I started,” “I found a small problem,” or “I could not finish.” Later, a human can read the diary and understand what happened—even if nobody was watching at the time.

This notebook preserves the original five-level and file-logging examples, then grows them into safe real-world patterns. It also reads the existing `app.py`, `app.log`, and `app1.log` examples without overwriting them.

## Learning map

| Level | Topics |
|---|---|
| Basic | why logging, five levels, messages, `basicConfig()` |
| Intermediate | named loggers, formatters, handlers, console + file, exceptions |
| Advanced | structured context, performance, testing, configuration habits |

## 1. Why logging instead of only `print()`?

| `print()` | Logging |
|---|---|
| quick value for a person now | lasting event record |
| no built-in severity | has levels |
| usually console only | console, files, network, and more |
| little context | time, module, line, process, message |
| awkward to turn on/off | configurable centrally |

Use `print()` for normal user-facing output. Use logging for developers/operators who need to understand program behavior.

## 2. The five common logging levels

Think of a traffic-light ladder. Higher numbers are more serious.

| Level | Number | Meaning | Example |
|---|---:|---|---|
| `DEBUG` | 10 | tiny diagnostic details | variable values during development |
| `INFO` | 20 | normal milestone | service started |
| `WARNING` | 30 | odd, but program continues | retrying a request |
| `ERROR` | 40 | one operation failed | file could not be read |
| `CRITICAL` | 50 | program may not continue | required database unavailable |

The configured level is a gate. At `WARNING`, `DEBUG` and `INFO` are ignored.

## 3. Preserve the original five-message example

In a notebook, logging may already have handlers installed. `force=True` resets the **root logger** so this teaching cell behaves predictably. Use `force=True` carefully in a library because it can replace an application's configuration.

In [ ]:
import logging

logging.basicConfig(level=logging.DEBUG,
    format="%(levelname)s | %(name)s | %(message)s",
    force=True,
)
# logging.basicConfig(level=logging.DEBUG)

logging.debug("This is a debug message")
logging.info("This is an info message")
logging.warning("This is a warning message")
logging.error("This is an error message")
logging.critical("This is a critical message")

DEBUG:root:This is a debug message
INFO:root:This is an info message
ERROR:root:This is an error message
CRITICAL:root:This is a critical message


### Step-by-step

1. `basicConfig()` creates a simple handler and formatter.
2. `level=logging.DEBUG` opens the gate for every standard level.
3. Each function makes a `LogRecord`.
4. The formatter turns the record into text.
5. The handler sends it to a destination (the console here).

By default, console logs go to `sys.stderr`, so notebook output may label them differently from `print()` output.

## 4. Named loggers and `__name__`

Real programs should usually use a logger named after the module:

```python
logger = logging.getLogger(__name__)
```

If a file is `shop/payments.py`, its imported module name might be `shop.payments`. This creates a useful family tree and tells us where each record came from.

In [2]:
logger = logging.getLogger(__name__)
logger.info("Notebook logger name is %s", logger.name)

INFO | __main__ | Notebook logger name is __main__


## 5. Message formatting: let logging do the work

Prefer placeholders for routine logs:

```python
logger.debug("User %s bought %d items", user_name, item_count)
```

Logging delays interpolation until it knows the record will be emitted. That can avoid wasted work. F-strings are readable but are evaluated even when the level is disabled.

In [3]:
user_name = "Asha"
item_count = 3
logger.info("User %s bought %d items", user_name, item_count)

INFO | __main__ | User Asha bought 3 items


## 6. Formatters: choose what each diary line contains

Useful fields include:

| Field | Meaning |
|---|---|
| `%(asctime)s` | event time |
| `%(name)s` | logger name |
| `%(levelname)s` | severity |
| `%(message)s` | final message |
| `%(module)s` / `%(funcName)s` | code location |
| `%(lineno)d` | source line number |

Never put secrets, passwords, tokens, or unnecessary personal data in logs.

In [4]:
formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
print("Formatter created:", formatter._fmt)

Formatter created: %(asctime)s | %(name)s | %(levelname)s | %(message)s


## 7. Handlers: delivery trucks for log records

| Handler | Destination |
|---|---|
| `StreamHandler` | console-like stream |
| `FileHandler` | one file |
| `RotatingFileHandler` | size-limited files |
| `TimedRotatingFileHandler` | files rotated by time |
| `NullHandler` | intentionally discards records, useful in libraries |

A logger can have multiple handlers. Each handler can have its own level and formatter.

## 8. File logging without touching repository logs

The original notebook used `filename='app.log'`, `filemode='w'`, and the five levels. We keep that behavior in a temporary directory. `w` replaces a file; `a` appends and is usually safer for application logs.

In [5]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporary_logs = TemporaryDirectory()
runtime_dir = Path(temporary_logs.name)
demo_log_path = runtime_dir / "app.log"

file_logger = logging.getLogger("tutorial.file")
file_logger.setLevel(logging.DEBUG)
file_logger.propagate = False
file_logger.handlers.clear()

file_handler = logging.FileHandler(demo_log_path, mode="w", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)
file_logger.addHandler(file_handler)

file_logger.debug("This is a debug message")
file_logger.info("This is an info message")
file_logger.warning("This is a warning message")
file_logger.error("This is an error message")
file_logger.critical("This is a critical message")

file_handler.flush()
print(demo_log_path.read_text(encoding="utf-8"))

2026-09-07 02:59:00 | tutorial.file | DEBUG | This is a debug message
2026-09-07 02:59:00 | tutorial.file | INFO | This is an info message
2026-09-07 02:59:00 | tutorial.file | WARNING | This is a warning message
2026-09-07 02:59:00 | tutorial.file | ERROR | This is an error message
2026-09-07 02:59:00 | tutorial.file | CRITICAL | This is a critical message



## 9. Console + file logging

Here the console gets `INFO` and above, while the file gets detailed `DEBUG` records. This is common: humans see clean status messages; developers retain more detail.

In [6]:
import sys

combined_logger = logging.getLogger("shop.checkout")
combined_logger.setLevel(logging.DEBUG)
combined_logger.propagate = False
combined_logger.handlers.clear()

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(logging.Formatter("CONSOLE | %(levelname)s | %(message)s"))

combined_path = runtime_dir / "combined.log"
combined_file_handler = logging.FileHandler(combined_path, mode="w", encoding="utf-8")
combined_file_handler.setLevel(logging.DEBUG)
combined_file_handler.setFormatter(formatter)

combined_logger.addHandler(console_handler)
combined_logger.addHandler(combined_file_handler)

combined_logger.debug("Coupon rule checked")
combined_logger.info("Checkout completed")
combined_logger.warning("Only one item remains")
combined_file_handler.flush()

print("\nFILE CONTENTS")
print(combined_path.read_text(encoding="utf-8"))

CONSOLE | INFO | Checkout completed


CONSOLE | WARNING | Only one item remains



FILE CONTENTS
2026-09-07 02:59:00 | shop.checkout | DEBUG | Coupon rule checked
2026-09-07 02:59:00 | shop.checkout | INFO | Checkout completed
2026-09-07 02:59:00 | shop.checkout | WARNING | Only one item remains



### Logger level and handler level both matter

A record must pass **two doors**:

1. The logger's level decides whether to create/pass the record.
2. Each handler's level decides whether that destination receives it.

The effective threshold is therefore at least as strict as both relevant gates.

## 10. Logging exceptions correctly

Inside an `except` block, `logger.exception(...)` records an `ERROR` message **and the traceback**. It is clearer than manually joining `str(error)` and a traceback.

In [7]:
def divide(a, b):
    try:
        result = a / b
        combined_logger.debug("Dividing %s by %s produced %s", a, b, result)
        return result
    except ZeroDivisionError:
        combined_logger.exception("Could not divide %s by %s", a, b)
        return None

print("Result:", divide(20, 0))

CONSOLE | ERROR | Could not divide 20 by 0
Traceback (most recent call last):
  File "C:\Users\HP\AppData\Local\Temp\ipykernel_37524\3070605467.py", line 3, in divide
    result = a / b
             ~~^~~
ZeroDivisionError: division by zero


Result: None


### When to use which pattern

| Pattern | Use |
|---|---|
| `logger.error("...", exc_info=True)` | include traceback at an explicit level |
| `logger.exception("...")` | inside `except`; shorthand for error + traceback |
| `logger.warning("...", stack_info=True)` | current call stack, even without an exception |

Avoid logging an exception and then logging it again at every layer. Usually the layer that *handles* the failure should log it; otherwise re-raise and let an outer boundary log once.

## 11. Read the repository's existing practical examples

The repository contains:

- `app.py`: arithmetic functions with console + `app1.log` logging.
- `logs/logger.py`: a file-based root configuration.
- `logs/test.py`: imports that configuration and logs an addition.
- historical `.log` files produced by those examples.

We display short, read-only previews so the originals remain unchanged.

In [8]:
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "app.py").exists():
    candidate = NOTEBOOK_DIR / "Complete-Python-Bootcamp-main" / "12-Logging In Python"
    if candidate.exists():
        NOTEBOOK_DIR = candidate

for relative_name in ["app.py", "logs/logger.py", "logs/test.py"]:
    path = NOTEBOOK_DIR / relative_name
    print(f"\n--- {relative_name} ---")
    print("\n".join(path.read_text(encoding="utf-8").splitlines()[:12]))


--- app.py ---
import logging

## logging setting

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler("app1.log"),
        logging.StreamHandler()
    ]

--- logs/logger.py ---
## configuring logging
import logging

logging.basicConfig(
    filename='app.log',
    filemode='w',
    level=logging.DEBUG,
    format='%(asctime)s-%(name)s-%(levelname)s-%(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
    )

--- logs/test.py ---
from logger import logging

def add(a,b):
    logging.debug("The addition operation is taking place")
    return a+b

logging.debug("The addition function is called")
add(10,15)


In [9]:
for relative_name in ["app.log", "app1.log", "logs/app.log"]:
    path = NOTEBOOK_DIR / relative_name
    lines = path.read_text(encoding="utf-8").splitlines()
    print(f"{relative_name}: {len(lines)} historical lines")
    for line in lines[:2]:
        print(" ", line)

app.log: 10 historical lines
  2024-06-19 12:25:46-root-DEBUG-This is a debug message
  2024-06-19 12:25:46-root-INFO-This is an info message
app1.log: 14 historical lines
  2024-06-19 13:34:00 - ArithmethicApp - DEBUG - Subtracting 15 - 10 = 5
  2024-06-19 13:34:00 - ArithmethicApp - DEBUG - Multiplying 10 * 20 = 200
logs/app.log: 2 historical lines
  2024-06-19 12:29:57-root-DEBUG-The addition function is called
  2024-06-19 12:29:57-root-DEBUG-The addition operation is taking place


### What the existing `app.py` teaches

Its structure is valuable: one named arithmetic logger, reusable functions, and both console and file handlers. Improvements for production code would be:

- spell the name consistently (`ArithmeticApp`);
- use module logger `logging.getLogger(__name__)`;
- configure handlers only in the entry point, not imported library modules;
- call functions under `if __name__ == "__main__":`;
- use `logger.exception()` when a traceback is helpful;
- avoid logging sensitive function arguments.

## 12. Add useful context

Logs become much more helpful when they carry stable context such as an order ID. `LoggerAdapter` adds fields without repeating them in every message.

In [10]:
context_formatter = logging.Formatter(
    "%(levelname)s | order=%(order_id)s | %(message)s"
)
context_handler = logging.StreamHandler(sys.stdout)
context_handler.setFormatter(context_formatter)

context_logger = logging.getLogger("shop.orders")
context_logger.handlers.clear()
context_logger.addHandler(context_handler)
context_logger.setLevel(logging.INFO)
context_logger.propagate = False

order_logger = logging.LoggerAdapter(context_logger, {"order_id": "ORD-1042"})
order_logger.info("Payment accepted")

INFO | order=ORD-1042 | Payment accepted


## 13. Configuration and library best practices

### Applications

- Configure logging once, near the program entry point.
- Prefer `dictConfig()` or a configuration file when setup grows.
- Choose levels, destinations, retention, and redaction intentionally.
- Use UTC timestamps when systems span time zones.

### Libraries

- Create `logging.getLogger(__name__)`.
- Do not call `basicConfig()` on import.
- Do not add surprise file handlers.
- Optionally attach `logging.NullHandler()` for older integration patterns.

### Security

Never log passwords, API keys, access tokens, full payment data, or unnecessary personal information. Redact before logging.

## 14. Common mistakes

| Mistake | Result | Fix |
|---|---|---|
| calling `basicConfig()` repeatedly | later calls may do nothing | configure once; notebook demos may use `force=True` |
| handlers added every function call | duplicate lines | configure once and clear/guard handlers in demos |
| logging and re-raising at every layer | repeated traceback noise | log once at the handling boundary |
| using root logger everywhere | unclear ownership | use `getLogger(__name__)` |
| using `ERROR` for normal events | alert fatigue | choose the honest severity |
| f-strings for expensive debug work | wasted computation | use `%s` arguments or `isEnabledFor()` |
| secrets in messages | security incident | redact sensitive data |
| endless single log file | disk fills | rotate and retain intentionally |

## 15. Clean up notebook-owned handlers

Closing handlers releases file handles—especially important on Windows. We remove only the handlers created by this notebook, then delete its temporary directory.

In [11]:
for demo_logger in [file_logger, combined_logger, context_logger]:
    for handler in demo_logger.handlers[:]:
        handler.flush()
        handler.close()
        demo_logger.removeHandler(handler)

temporary_logs.cleanup()
print("Temporary tutorial logs cleaned up; repository logs were untouched.")

Temporary tutorial logs cleaned up; repository logs were untouched.


# Quick Revision Cheat Sheet

## Minimum good pattern

```python
import logging

logger = logging.getLogger(__name__)

def do_work(item_id):
    logger.info("Processing item %s", item_id)

if __name__ == "__main__":
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    )
    do_work("A-42")
```

## Remember

- Levels: `DEBUG < INFO < WARNING < ERROR < CRITICAL`.
- Logger creates/routes records; formatter shapes text; handler sends it somewhere.
- Use `getLogger(__name__)` in modules.
- Use `logger.exception()` inside `except` when a traceback helps.
- Configure once at the application boundary.
- Use lazy `%s` arguments, rotate long-lived files, and never log secrets.